# Subword Tokenization

In this exercise, we will learn how to train our own subword tokenizers with different algorithms: BPE and Unigram. We will use `sentencepiece`, a library from Google to help create our tokenizers.

## Ref:
https://github.com/google/sentencepiece/blob/master/python

## Setup

In [3]:
!wget https://github.com/Knight-H/thai-lm/raw/refs/heads/master/data/pra-apai-manee-ch1-50.txt
!wget https://github.com/Knight-H/thai-lm/raw/refs/heads/master/data/kratoo-40000000-40002000.jsonl

'wget' is not recognized as an internal or external command,
operable program or batch file.
'wget' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!pip install sentencepiece

   ---------------------------------------- 0.0/991.5 kB ? eta -:--:--
   --------------------------------------- 991.5/991.5 kB 23.5 MB/s eta 0:00:00


## Code

In [8]:
import sentencepiece as spm
import io
import json

Load data

In [10]:
pantip_text = []
with open(r"C:\Users\WINDOWS 11\OneDrive\เดสก์ท็อป\NLP\HW2\kratoo-40000000-40002000.jsonl", 'r', encoding='utf-8') as json_file:
    json_list = list(json_file)
    for json_str in json_list:
        result = json.loads(json_str)
        pantip_text.append(f"{result['title']}\n{result['content']}\n")
sum([len(t) for t in pantip_text])

1060318

In [13]:
with open(r"C:\Users\WINDOWS 11\OneDrive\เดสก์ท็อป\NLP\HW2\pra-apai-manee-ch1-50.txt", encoding='utf-8') as f:
  pra_apai_manee_data = f.readlines()

In [14]:
sum([len(t) for t in pra_apai_manee_data])

1100605

In [15]:
pantip_train_text = pantip_text[:int(len(pantip_text)*0.8)]
pantip_test_text = pantip_text[int(len(pantip_text)*0.8):]

pam_train_text = pra_apai_manee_data[:int(len(pra_apai_manee_data)*0.8)] #pam = pra_apai_manee
pam_test_text = pra_apai_manee_data[int(len(pra_apai_manee_data)*0.8):]

## Run tokenizer training

The Python wrapper provides multiple APIs for training our tokenizers

1. `spm.SentencePieceTrainer.train(input='input.txt', model_prefix='m', vocab_size=vocab_size, model_type=model_type)`
  <br> This will output the tokenizer files `m.model` and `m.vocab` that can be later loaded into `SentencePieceProcessor`.
  <br><br>
2. `spm.SentencePieceTrainer.train(sentence_iterator=iterator, model_writer=obj_with_write_method, vocab_size=vocab_size, model_type=model_type)`
  <br> This method will require a file object e.g. `obj_with_write_method = io.BytesIO()`. The advantage of this method is you can run sentencepiece on environments that have limited access to the local file system. But you will still have to save the model file if you want to re-use the model else you will have to train it again.
<br><br>
3.  `spm.SentencePieceTrainer.train('--input=input.txt --model_prefix=m --vocab_size=vocab_size --model_type=model_type')`
<br> Same as no.1




### Unigram tokenizer

We are going to start with training a unigram tokenizer. You can use any method of training one. Make sure to set vocab_size to 1000.

In [17]:
## Train
model_buffer = io.BytesIO()
vocab_size = 1000
model_type = "unigram"

sentence_iterator = iter(pam_train_text)

spm.SentencePieceTrainer.train(
    sentence_iterator=sentence_iterator,
    model_writer=model_buffer,
    vocab_size=vocab_size,
    model_type=model_type
)

sp_pam = spm.SentencePieceProcessor()
sp_pam.LoadFromSerializedProto(model_buffer.getvalue())

True

### Q1 MCV

How many tokens did you get when tokenizing the following sentence with your unigram tokenizer: <br>
'อรุณสวัสดิ์ ฉันเอามเหสีมาหาม สวัสดี ประเทศไทยสบายดีไหม'

In [18]:
len(sp_pam.encode('อรุณสวัสดิ์ ฉันเอามเหสีมาหาม สวัสดี ประเทศไทยสบายดีไหม', out_type=str))

29

### BPE Tokenizer

Now try training a BPE tokenizer.

In [26]:
vocab_size = 1000
model_type = "bpe"

model_buffer = io.BytesIO()

sentence_iterator = iter(pam_train_text)

spm.SentencePieceTrainer.train(
    sentence_iterator=sentence_iterator,
    model_writer=model_buffer,
    vocab_size=vocab_size,
    model_type=model_type
)

bpe = spm.SentencePieceProcessor()
bpe.LoadFromSerializedProto(model_buffer.getvalue())

True

### Q2 MCV

How many tokens did you get when tokenizing the following sentence with your BPE tokenizer: <br>
'อรุณสวัสดิ์ ฉันเอามเหสีมาหาม สวัสดี ประเทศไทยสบายดีไหม'

In [27]:
len(bpe.encode('อรุณสวัสดิ์ ฉันเอามเหสีมาหาม สวัสดี ประเทศไทยสบายดีไหม', out_type=str))

28

These are some of your vocabs. Note that you will see "▁" (U+2581) in every type of tokenizer in SentencePiece since it makes it possible to perform detokenization \(unsplit your sentences\) without relying on language-specific resources.

In [28]:
unigram_vocabs = [sp_pam.id_to_piece(id) for id in range(sp_pam.get_piece_size())]
" | ".join(unigram_vocabs[:500])

'<unk> | <s> | </s> | ▁ | า | เ | น | ม | ย | ก | ร | ว | ด | ส | ง | บ | ค | มา | อ | ล | จะ | ท | ให้ | ห | ไป | ไม่ | แ | ว่า | พ | ุ | ี | ๏ | ฯ | ข | ช | เป็น | พระ | โ | ที่ | ใจ | ▁จะ | จ | ะ | ิ | ต | ก็ | อยู่ | ป | ได้ | ่ | ไ | เข้า | ู | ▁พระ | ้า | ตาม | ใน | ้ | ▁แล้ว | เหมือน | รา | ศ | เจ้า | เห็น | ลา | กัน | ั | หา | นาง | ทรง | ประ | ์ | ยา | ัก | ํา | ซ | าน | ัง | ฉ | องค์ | ัด | แล้ว | อน | ดู | ถ | ด้วย | มี | ▁จึง | นี้ | ่า | ผ | น้อง | แต่ | ทํา | ▁นาง | ▁ให้ | รัก | พี่ | คิด | ลูก | พา | รู้ | การ | กับ | ัน | หน้า | กระ | วน | ออก | ่อ | เขา | ถึง | ระ | ข้า | ับ | พล | นั่ง | ทั้ง | หน | รับ | ษ | กล | วง | ลง | ฝ | กร | พร | ความ | เสีย | ดี | ขึ้น | อง | ่ง | ธ | ▁แต่ | คน | กลับ | ▁ฝ่าย | ้น | อด | ภ | หรือ | ตร | ือ | ฟัง | แม่ | ▁ไม่ | ไว้ | ยัง | ▁เห็น | นา | ขอ | มิ | น้ํา | หล | ดัง | ▁พอ | ▁ทั้ง | ช่วย | สม | นั้น | ริ | ทัพ | ต้อง | วัน | อา | น้อย | รบ | ิน | อย่า | เอา | จน | เรา | สุด | เสียง | ข้าง | หลัง | ตี | ตัว | ละ | สุ | วัง | ทุก | ่น

In [29]:
bpe_vocabs = [bpe.id_to_piece(id) for id in range(bpe.get_piece_size())]
" | ".join(bpe_vocabs[:500])

'<unk> | <s> | </s> | ้า | ่า | อง | ระ | ํา | รา | อย | ่ง | มา | จะ | ัง | ัน | ▁เ | าย | ้ว | ับ | ี่ | ม่ | อน | ให | าม | ้น | ็น | พระ | ีย | าง | กล | ้ง | ัก | หน | ให้ | ไม่ | หล | ่น | ึง | ▁แ | ทั | ตร | าร | ้อง | ไป | ิด | ข้า | ว่า | หม | คร | ือ | ล้ว | เป | เส | ประ | าน | ั่ง | ▁๏ | ▁ฯ | ที่ | อก | เล | ิน | ได | พล | ทร | ัด | นาง | ึก | ได้ | ู่ | ▁จะ | ค์ | ี้ | พร | เป็น | สุ | ทั้ง | อม | ัย | เร | ห็น | ▁จ | ▁พระ | ก็ | ใจ | อา | ื่ | ่าง | ต่ | กร | ิง | วง | วน | ือน | เจ | ู้ | ียง | อยู่ | รร | ตาม | ▁พ | ้วย | าว | ถึง | คล | ั้น | รี | เข | ด้วย | สม | องค์ | สน | าก | ▁แล้ว | เช | ัว | ย์ | ใน | คว | น้ | หมือน | ▁ส | ูก | อบ | กระ | เจ้า | ทรง | ลา | กัน | มี | ่าย | พรา | ิ่ง | เข้า | เห็น | ิต | สง | อด | ณ์ | วย | ้ม | คิด | เม | เก | เด | ▁นาง | วา | ุก | ▁ให้ | ดู | หา | ▁อ | ▁จึง | ทํา | ลง | รัก | เค | แล้ว | ่าน | พี่ | เหมือน | ั่น | ความ | ยง | อย่า | หร | มิ | ืน | ช่ | การ | ัญ | ▁ไม่ | ฝ่าย | ศรี | ้าง | วก | ้อม | ือง | น้อง | ยว | พา | แก |

### User-defined symbols

Another important concept to know of is User-defined symbols. These special symbols are reserved for a special purpose \(e.g.\, the \<MASK\> token used in BERT) and will always be tokenized into one token.

Refer to the documentation for ways to add these special tokens to your tokenizer.

https://github.com/google/sentencepiece/blob/master/python

## Train another tokenizer on another domain

Now try training another unigram tokenizer on `pantip_text` and we will use it to compare with the unigram tokenizer we trained earlier.

In [31]:
## Train
vocab_size = 1000
model_type = "unigram"

# สร้าง in-memory buffer สำหรับเก็บโมเดลที่เทรน
model_buffer = io.BytesIO()

# เทรน Unigram tokenizer บนข้อมูล pantip_text
spm.SentencePieceTrainer.train(
    sentence_iterator=iter(pantip_text),
    model_writer=model_buffer,
    vocab_size=vocab_size,
    model_type=model_type
)

# โหลดโมเดลจากหน่วยความจำโดยตรง
sp_pantip = spm.SentencePieceProcessor()
sp_pantip.LoadFromSerializedProto(model_buffer.getvalue())

True

## Analyse top tokens on different datasets

Use your tokenizers to tokenize the datasets and analyse your most common vocabularies (try 300-400 vocabs with len>1). Hint: tokenize your data and count the tokens.

In [32]:
from collections import Counter

# For pra-apai-manee tokenizer (sp_pam)
# Assuming pam_train_text is your list of training sentences from pra-apai-manee
token_counts_pam = Counter()

for sentence in pam_train_text:
    tokens = sp_pam.encode(sentence, out_type=str)
    # Filter out tokens with length 1 or less
    filtered_tokens = [token for token in tokens if len(token) > 1]
    token_counts_pam.update(filtered_tokens)

# Get top 300 tokens for pra-apai-manee
top_tokens_pam = token_counts_pam.most_common(300)
print("Top tokens in pra-apai-manee dataset (tokenizer: sp_pam):")
for token, count in top_tokens_pam:
    print(f"{token}: {count}")

print("\n" + "="*40 + "\n")

# For Pantip tokenizer (sp_pantip)
# Assuming pantip_text is your list of sentences from Pantip
token_counts_pantip = Counter()

for sentence in pantip_text:
    tokens = sp_pantip.encode(sentence, out_type=str)
    filtered_tokens = [token for token in tokens if len(token) > 1]
    token_counts_pantip.update(filtered_tokens)

# Get top 300 tokens for Pantip
top_tokens_pantip = token_counts_pantip.most_common(300)
print("Top tokens in Pantip dataset (tokenizer: sp_pantip):")
for token, count in top_tokens_pantip:
    print(f"{token}: {count}")

Top tokens in pra-apai-manee dataset (tokenizer: sp_pam):
มา: 3002
จะ: 2506
ไป: 2407
ให้: 2391
ว่า: 2214
ไม่: 2085
ที่: 1632
เป็น: 1587
▁จะ: 1580
พระ: 1561
ใจ: 1458
ก็: 1368
อยู่: 1339
▁พระ: 1251
ได้: 1247
เข้า: 1190
รา: 1128
ตาม: 1096
้า: 1086
▁แล้ว: 1085
ใน: 1069
ยา: 1063
ลา: 1052
ํา: 1039
เจ้า: 1024
เหมือน: 1009
กัน: 976
หา: 973
ประ: 944
อน: 933
ทรง: 914
ัง: 914
เห็น: 910
▁ให้: 898
าน: 879
นาง: 875
ัก: 868
ัด: 837
ดู: 834
องค์: 827
มี: 809
่า: 801
▁จึง: 799
▁นาง: 792
วน: 782
พา: 777
แล้ว: 771
นี้: 770
่อ: 765
ด้วย: 760
ลูก: 744
น้อง: 741
ทํา: 738
รัก: 736
หน: 731
พี่: 719
การ: 714
คิด: 712
พล: 712
กระ: 710
รู้: 695
กับ: 683
แต่: 677
หน้า: 670
ระ: 666
ออก: 665
▁ไม่: 662
ัน: 660
ับ: 656
เขา: 652
่ง: 637
พร: 622
นั่ง: 620
ข้า: 620
คน: 617
รับ: 616
วง: 614
ดี: 610
ถึง: 601
ริ: 597
▁แต่: 592
หล: 585
▁เห็น: 581
ลง: 578
กร: 576
อด: 576
กล: 574
สม: 573
ความ: 573
ตร: 568
ทั้ง: 566
ขึ้น: 566
เสีย: 564
ือ: 557
กลับ: 555
▁ฝ่าย: 553
ปร: 544
มิ: 540
ขอ: 540
้น: 539
หรือ: 534
ยัง: 531
▁ทั้ง: 528
เ

### To answer
What are some notable differences you see between the two vocabs?

Write your answer below.

คำใน pantip เยอะกว่า และมีภาษาอังกฤษปนอยู่

## Using tokenizer across domains

One problem you may face is your dataset is very specialized. In that case the tokenizer trained on a general domain may not perform as good as it should when used on your dataset.

Next you will try using tokenizers trained on one general domain (on Pantip) and use it on a specialized domain (พระอภัยมณี) and vice versa.

### Q3 MCV

What percentage increase do you observe when tokenizing the whole พระอภัยมณี dataset with a tokenizer trained on Pantip compared to the one trained on พระอภัยมณี.

In [35]:
total_tokens_pam = 0
total_tokens_pantip = 0

for sentence in pra_apai_manee_data:
    total_tokens_pam += len(sp_pam.encode(sentence, out_type=str))
    total_tokens_pantip += len(sp_pantip.encode(sentence, out_type=str))

# คำนวณเปอร์เซ็นต์เพิ่ม: (tokens ที่ได้จาก Pantip - tokens ที่ได้จาก พระอภัยมณี) / tokens ที่ได้จาก พระอภัยมณี * 100
percentage_increase = (total_tokens_pantip - total_tokens_pam) / total_tokens_pam * 100

print("Percentage Increase:", percentage_increase, "%")


Percentage Increase: 44.92998881524534 %


### Q4 MCV

What percentage increase do you observe when tokenizing the whole Pantip dataset with a tokenizer trained on พระอภัยมณี compared to the one trained on Pantip.

In [36]:
total_tokens_pantip_specialized = 0
total_tokens_pantip_general = 0

for sentence in pantip_text:
    total_tokens_pantip_specialized += len(sp_pam.encode(sentence, out_type=str))
    total_tokens_pantip_general += len(sp_pantip.encode(sentence, out_type=str))

percentage_increase = (total_tokens_pantip_specialized - total_tokens_pantip_general) / total_tokens_pantip_general * 100
print("Percentage Increase:", percentage_increase, "%")


Percentage Increase: 12.778124373097247 %


### To answer
Why do you think the number of tokens tokenized by the general tokenizer (the one trained on Pantip) has a higher percentage increase compared to the number of tokens tokenized by the specialized tokenizer? (Hint: we fixed vocab size.)

เพราะว่าเมื่อใช้ tokenizer ที่เทรนจากข้อมูลทั่วไป (Pantip) กับโดเมนเฉพาะอย่างพระอภัยมณี จะไม่มีคำหรือ subword ที่เหมาะกับโดเมนนั้นอยู่ใน vocab ทำให้ต้องแบ่งคำออกเป็น subword เล็กๆ เพิ่มขึ้น ส่งผลให้จำนวน token สูงขึ้นเมื่อเทียบกับการใช้ tokenizer ที่เทรนเฉพาะโดเมนแล้ว

## The effect on language models

Next, we will see the effect of using "cross-domain" tokenizers on Language models.

### Setup
We are going to reuse the code from the last assignment

In [37]:
!pip install lightning

In [38]:
import itertools
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import lightning as L
from tqdm import tqdm
import numpy as np

In [39]:
class TextDataset(Dataset):
  def __init__(self, data, tokenizer, seq_len = 128):

    token_ids = [tokenizer.encode(d, add_bos=True, add_eos=True) for d in data]
    flatten_token_ids = list(itertools.chain(*token_ids))
    encoded = torch.LongTensor(flatten_token_ids)

    left_over = len(encoded) % seq_len
    encoded = encoded[:len(encoded)-left_over]
    self.encoded = encoded.view(-1, seq_len)

  def __getitem__(self, idx):
    return self.encoded[idx]

  def __len__(self):
    return len(self.encoded)

In [44]:
class LSTM(L.LightningModule):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, learning_rate, criterion):

        super().__init__()

        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.embedding_dim = embedding_dim
        self.vocab_size=vocab_size

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers,
                    dropout=dropout_rate, batch_first=True)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.learning_rate = learning_rate
        self.criterion = criterion

    def forward(self, src):
        # src: [batch_size, seq_len]
        x = self.embedding(src)              # x: [batch_size, seq_len, embedding_dim]
        output, _ = self.lstm(x)             # output: [batch_size, seq_len, hidden_dim]
        output = self.dropout(output)
        logits = self.fc(output)             # logits: [batch_size, seq_len, vocab_size]
        return logits


    def training_step(self, batch, batch_idx):

        src = batch[:, :-1]
        target = batch[:, 1:]
        prediction = self(src)
        prediction = prediction.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        loss = self.criterion(prediction, target)
        self.log("train_loss", loss)
        return loss

    def test_step(self, batch, batch_idx, dataloader_idx=0):

        src = batch[:, :-1]
        target = batch[:, 1:]
        with torch.no_grad():
          prediction = self(src)
        prediction = prediction.reshape(-1, self.vocab_size)
        target = target.reshape(-1)
        loss = self.criterion(prediction, target)
        self.log("test_loss", loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.learning_rate)

In [45]:
vocab_size = sp_pam.get_piece_size()
embedding_dim = 200
hidden_dim = 512
num_layers = 3
dropout_rate = 0.2
lr = 1e-3
criterion = nn.CrossEntropyLoss()
train_batch_size = 64
test_batch_size = 128

### Training

<a name="no1"></a>
#### 1. Training on Pantip data with Pantip tokenizer

In [46]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pantip)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pantip)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pantip)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pantip)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pantip_train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.511    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 46/46 [00:02<00:00, 20.16it/s, v_num=2]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 46/46 [00:02<00:00, 18.07it/s, v_num=2]


In [47]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\WINDOWS 11\anaconda3\envs\NLP\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:476: Your `test_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
C:\Users\WINDOWS 11\anaconda3\envs\NLP\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing DataLoader 3: 100%|██████████| 9/9 [00:00<00:00, 35.93it/s]  
Perplexity on Pantip train set is:	78.57598543715875
Perplexity on Pra apai manee train set is:	112.08904436142613
Perplexity on Pantip test set is:	117.66682039501498
Perplexity on Pra apai manee test set is:	113.95510531463852


<a name="no2"></a>
#### 2. Training on Pantip data with Pra apai manee tokenizer

In [48]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pam)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pam)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pam)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pam)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pantip_train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.511    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 51/51 [00:02<00:00, 18.79it/s, v_num=3]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 51/51 [00:02<00:00, 17.07it/s, v_num=3]


In [49]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"PerplAexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 3: 100%|██████████| 7/7 [00:00<00:00, 35.33it/s]  
Perplexity on Pantip train set is:	39.95265093765702
Perplexity on Pra apai manee train set is:	488.1733119612987
Perplexity on Pantip test set is:	50.35239391716569
Perplexity on Pra apai manee test set is:	458.52116786827895


#### To answer

The perplexity numbers should indicate that:
1. Training the LM with Pra apai manee tokenizer on Pantip (no. [2](#no2)) results in overfitting to Pantip and poor generalization to the Pra apai manee dataset.
2. However using the Pantip tokenizer (no. [1](#no1)) results in a much better generalization.

Try and come up with some reasons for the results above. <br>
Hint:
1. think about "general" vocabs and domain-specific vocabs.
2. what do you think happens to the model when the token ids become longer.

การใช้ tokenizer ที่เทรนบนพระอภัยมณีมี vocab เฉพาะโดเมน ทำให้เมื่อใช้กับ Pantip เกิดการแบ่งคำที่ละเอียดเกินไป โมเดลจึง overfit และ generalize ได้ไม่ดีเท่า tokenizer จาก Pantip ซึ่งมี vocab ที่ครอบคลุมหลายโดเมน


<a name="no3"></a>
#### 3. Training on Pra apai manee data with Pantip tokenizer


In [50]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pantip)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pantip)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pantip)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pantip)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pam_train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Epoch 0:   0%|          | 0/46 [10:23<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.511    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode


Epoch 9: 100%|██████████| 67/67 [00:03<00:00, 18.52it/s, v_num=4]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 67/67 [00:03<00:00, 17.15it/s, v_num=4]


In [51]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 3: 100%|██████████| 9/9 [00:00<00:00, 36.30it/s]  
Perplexity on Pantip train set is:	4918.501179471426
Perplexity on Pra apai manee train set is:	39.7889588695451
Perplexity on Pantip test set is:	4628.959160314484
Perplexity on Pra apai manee test set is:	42.73456092731207


<a name="no4"></a>
#### 4. Training on Pra apai manee data with Pra apai manee tokenizer




In [52]:
trainer = L.Trainer(
    max_epochs=10,
    deterministic=True
)
model = LSTM(vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, lr, criterion)

pantip_train_dataset = TextDataset(pantip_train_text, sp_pam)
pantip_train_loader = DataLoader(pantip_train_dataset, batch_size = train_batch_size, shuffle = True)

pantip_test_dataset = TextDataset(pantip_test_text, sp_pam)
pantip_test_loader = DataLoader(pantip_test_dataset, batch_size = test_batch_size, shuffle = False)

pam_train_dataset = TextDataset(pam_train_text, sp_pam)
pam_train_loader = DataLoader(pam_train_dataset, batch_size = train_batch_size, shuffle = True)

pam_test_dataset = TextDataset(pam_test_text, sp_pam)
pam_test_loader = DataLoader(pam_test_dataset, batch_size = test_batch_size, shuffle = False)

trainer.fit(model, train_dataloaders=pam_train_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode 
-------------------------------------------------------
0 | embedding | Embedding        | 200 K  | train
1 | lstm      | LSTM             | 5.7 M  | train
2 | dropout   | Dropout          | 0      | train
3 | fc        | Linear           | 513 K  | train
4 | criterion | CrossEntropyLoss | 0      | train
-------------------------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total params
25.511    Total estimated model params size (MB)
5         Modules in train mode
0         Modules in eval mode
C:\Users\WINDOWS 11\anaconda3\envs\NLP\Lib\site-packages\lightning\pytorch\loops\fit_loop.py:310: The number of training batches (48) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_

Epoch 9: 100%|██████████| 48/48 [00:02<00:00, 19.37it/s, v_num=5]

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 48/48 [00:02<00:00, 17.45it/s, v_num=5]


In [53]:
test_result = trainer.test(model, dataloaders=[pantip_train_loader, pam_train_loader, pantip_test_loader,pam_test_loader], verbose=False)

print(f"Perplexity on Pantip train set is:\t{np.exp(test_result[0]['test_loss/dataloader_idx_0'])}")
print(f"Perplexity on Pra apai manee train set is:\t{np.exp(test_result[1]['test_loss/dataloader_idx_1'])}")
print(f"Perplexity on Pantip test set is:\t{np.exp(test_result[2]['test_loss/dataloader_idx_2'])}")
print(f"Perplexity on Pra apai manee test set is:\t{np.exp(test_result[3]['test_loss/dataloader_idx_3'])}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 3: 100%|██████████| 7/7 [00:00<00:00, 34.70it/s]  
Perplexity on Pantip train set is:	515.1974514812767
Perplexity on Pra apai manee train set is:	72.55858207639663
Perplexity on Pantip test set is:	507.65090954407566
Perplexity on Pra apai manee test set is:	82.84597223922799


#### To answer

The perplexity numbers should indicate that:
1. Both LM overfits on Pra apai manee data and performs really bad on Pantip data.
2. However using the Pra apai manee tokenizer (no. [4](#no4)) results in a  better generalization than the Pantip tokenizer(no. [3](#no3)).

Try and come up with some reasons for the results above. <br>

1. LM ที่ฝึกบนข้อมูลพระอภัยมณี โมเดลมัก overfit กับสไตล์โดเมนเฉพาะ ทำให้ประสิทธิภาพแย่ลงเมื่อทดสอบกับ Pantip

2. เมื่อใช้ Pra apai manee tokenizer (no.4) แม้จะ overfit บนข้อมูลเฉพาะโดเมน แต่การแบ่งคำที่แม่นยำและเหมาะสมกับสไตล์ของพระอภัยมณี ทำให้ token sequence สั้นและมีความสอดคล้องกันมากขึ้น ส่งผลให้โมเดล generalize ได้ดีกว่าเมื่อเทียบกับการใช้ Pantip tokenizer (no.3) ซึ่งแบ่งคำไม่ตรงกับโดเมน ทำให้เกิด token sequence ที่ยาวและประสิทธิภาพแย่ลงเมื่อทดสอบกับ Pantip
